In [1]:
import polars as pl
from pathlib import Path
import os
from sklearn.preprocessing import StandardScaler, RobustScaler, PowerTransformer, MinMaxScaler
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

In [2]:
data_folder = Path.cwd().parents[1]/'results'
parquet_files = list(data_folder.glob('*.parquet'))

all_df = {filename.name.split('.')[0].split('_')[-1]: pl.scan_parquet(filename) for filename in parquet_files}

In [3]:
# Tag each row with the policy (file) it came from, so shifts can stay within one run.
main = pl.concat([lf.with_columns(pl.lit(name).alias('policy')) for name, lf in all_df.items()])

## input features for RL

demand forecast     
previous days price     
in-hand stock       
buying cost: treated as simple average   

## input scalars

In [4]:
products = list(main.select(
    pl.col('product').unique()
).sort(by='product').collect())[-1]

In [5]:
# State = what is known before pricing day t: yesterday's price, demand and closing stock
# (today's demand and end-of-day stock would leak the day being priced).
# Price is scaled by CustomScaler.price_scaler (DP's price range -> [-1, 1], same as the action),
# so only demand and stock are fitted here.
w = ('policy', 'epoch', 'product')
for product in products:

    input = main.select(
        pl.col('day','product'),
        pl.col('demand').shift(1).over(*w, order_by='day').alias('yesterday_demand'),
        pl.col('stock').shift(1).over(*w, order_by='day').alias('yesterday_stock'),
    ).filter(
        pl.col('day')!=0,
        pl.col('product')==product
    ).drop('day','product')

    x = input.collect().to_numpy()
    pt = PowerTransformer()
    pt.fit(x)
    input_scaler = RobustScaler()
    input_scaler.fit(pt.transform(x))

    joblib.dump(pt, Path.cwd().parents[1]/'code'/'work'/'RL'/'scaler'/f'input_pt_{product}.pkl')
    joblib.dump(input_scaler, Path.cwd().parents[1]/'code'/'work'/'RL'/'scaler'/f'input_rs_{product}.pkl')

## reward feedback

positive feedback: (revenue/items sold)       
negative feedback: wastage

In [6]:
for product in products:

    output = main.filter(
        pl.col('product')==product,                       
    ).with_columns(
        (pl.col('order_cost')/pl.col('order_qty'))
            .forward_fill().over('policy', 'epoch', order_by='day').alias('cost'),
    ).select(
        (pl.col('sold')*pl.col('price')).alias('revenue'),
        (pl.col('cost')*pl.col('wastage')).alias('wastage_cost'),
    ).drop_nulls()

    data = output.collect().to_numpy()
    scale = data.max()                        
    output_scaler = MinMaxScaler().fit([[0, 0], [scale, scale]])

    joblib.dump(output_scaler, Path.cwd().parents[1]/'code'/'work'/'RL'/'scaler'/f'output_mm_{product}.pkl')

## import scaler

In [7]:
import sys
scaler_dir = str(Path.cwd().parents[1]/'code'/'work'/'RL')
if scaler_dir not in sys.path:
    sys.path.insert(0, scaler_dir)

from scaler.custom_scaler import CustomScaler
import PPO

scaler = CustomScaler()

# pretraining

In [8]:
import torch as T
import torch.nn.functional as F
import numpy as np

In [ ]:
class SampleData:
    """Pretraining data drawn from several policies' logs.

    sample_frac: {policy: fraction of that policy's runs (epochs) to use}, e.g.
                 {'Daily': 0.3, 'SinglePrice': 0.1}. Runs are picked at random, and every
                 picked run is used in full (all its days), so the data contains both the
                 good days (revenue) and the bad ones (expiry waste).
    For each product it builds aligned arrays:
        state   scaled [yesterday's price, demand, closing stock]   -> actor & critic input
        action  scaled price the policy chose that day               -> actor target
        return  discounted sum of the run's future rewards           -> critic target
    and serves them as shuffled minibatches.
    """

    def __init__(self, main, scaler, sample_frac, gamma=0.99, seed=0):
        self.scaler, self.gamma = scaler, gamma
        self.rng = np.random.default_rng(seed)
        w = ('policy', 'epoch', 'product')

        # 1. pick runs at random, per policy
        runs = []
        for policy, frac in sample_frac.items():
            epochs = np.sort(main.filter(pl.col('policy') == policy)
                             .select(pl.col('epoch').unique()).collect()['epoch'].to_numpy())
            if len(epochs) == 0:
                raise ValueError(f'no runs for policy {policy!r}')
            k = max(1, int(round(frac * len(epochs))))
            runs += [(policy, int(e)) for e in self.rng.choice(epochs, size=k, replace=False)]
        self.runs = pl.DataFrame(runs, schema={'policy': pl.Utf8, 'epoch': pl.Int32}, orient='row')

        # 2. the full log of each picked run, with state, cost and reward columns
        self.frame = (main
            .join(self.runs.lazy(), on=['policy', 'epoch'], how='inner')
            .with_columns(
                pl.col('price').shift(1).over(*w, order_by='day').alias('yesterday_price'),
                pl.col('demand').shift(1).over(*w, order_by='day').alias('yesterday_demand'),
                pl.col('stock').shift(1).over(*w, order_by='day').alias('yesterday_stock'),
                # unit cost of the latest order; days before the run's first order take the
                # first order's cost, so early waste is still costed
                (pl.col('order_cost') / pl.col('order_qty'))
                    .forward_fill().backward_fill().over(*w, order_by='day').alias('cost'),
            )
            .with_columns(
                (pl.col('sold') * pl.col('price')).fill_null(0.0).alias('revenue'),   # unpriced day: sold 0
                (pl.col('cost') * pl.col('wastage')).fill_null(0.0).alias('wastage_cost'),
            )
            .filter(pl.col('day') > 0)                         # day 0 has no "yesterday"
            .sort('product', 'policy', 'epoch', 'day')
            .collect())
        self.products = self.frame['product'].unique().sort().to_list()
        self.data = {p: self._build(p) for p in self.products}

    def _build(self, p):
        d = self.frame.filter(pl.col('product') == p)
        reward = self.scaler.output_scaler(p, d.select('revenue', 'wastage_cost').to_numpy()).sum(axis=1)

        # discounted return over every day of the run, restarting at each run boundary
        run_id = (d['policy'] + '_' + d['epoch'].cast(pl.Utf8)).to_numpy()
        ret, g = np.zeros(len(reward), dtype=np.float32), 0.0
        for t in reversed(range(len(reward))):
            if t == len(reward) - 1 or run_id[t + 1] != run_id[t]:
                g = 0.0
            g = reward[t] + self.gamma * g
            ret[t] = g

        # training rows need a full state and the price chosen that day
        ok = (d.select(pl.all_horizontal(pl.col('yesterday_price', 'yesterday_demand',
                                                 'yesterday_stock', 'price').is_not_null()))
               .to_series().to_numpy())
        d = d.filter(pl.Series(ok))
        return {
            'state':  self.scaler.input_scaler(p, d.select('yesterday_price', 'yesterday_demand', 'yesterday_stock').to_numpy()),
            'action': self.scaler.price_scaler(p, d['price'].to_numpy()),
            'reward': reward[ok].astype(np.float32),
            'return': ret[ok],
            'policy': d['policy'].to_numpy(),
            'epoch':  d['epoch'].to_numpy(),
        }

    def summary(self):
        """Rows, share of negative-reward days and reward range per product and policy."""
        rows = []
        for p, d in self.data.items():
            for pol in np.unique(d['policy']):
                m = d['policy'] == pol
                r = d['reward'][m]
                rows.append((p, pol, int(m.sum()), float((r < 0).mean()), float(r.min()), float(r.max())))
        return pl.DataFrame(rows, schema=['product', 'policy', 'rows', 'neg_share', 'reward_min', 'reward_max'], orient='row')

    def minibatches(self, product, n_passes=1, batch_size=256):
        """Shuffled passes over all rows of one product: yields (state, action, return)."""
        d = self.data[product]
        for _ in range(n_passes):
            perm = self.rng.permutation(len(d['state']))
            for i in range(0, len(perm), batch_size):
                idx = perm[i:i + batch_size]
                yield d['state'][idx], d['action'][idx], d['return'][idx]

    def sample(self, product, batch_size=256):
        """One random minibatch (with replacement): (state, action, return)."""
        d = self.data[product]
        idx = self.rng.integers(0, len(d['state']), batch_size)
        return d['state'][idx], d['action'][idx], d['return'][idx]

In [10]:
def to_action(product, price):
    return scaler.price_scaler(product, price)

def discounted_returns(r, gamma):
    ret, g = np.zeros(len(r), dtype=np.float32), 0.0
    for t in reversed(range(len(r))):
        if t == len(r)-1:
            g = 0.0
        g = r[t] + gamma * g
        ret[t] = g
    return ret        

In [ ]:
# fraction of each policy's runs to use; every picked run is used in full
sample_frac = {
    'Daily':       0.1,
    'SinglePrice': 0.1,
    '1Markdown':   0.1,
    'MarginPrice': 0.1,
    'ConstantPrice': 0.1,
}
sd = SampleData(main, scaler, sample_frac, gamma=0.99)
sd.summary()

In [ ]:
agents = {p: PPO.Agent(n_actions=1, input_dims=3, action_low=[-1.0], action_high=[1.0],
                       gamma=0.99, alpha=3e-4, gae_lambda=0.95, batch_size=64, n_epochs=10)
          for p in products}

def pretrain(p, agent, n_passes=20, batch_size=256, log_every=5):
    
    dev = agent.actor.device
    for e in range(n_passes):
        bc_sum = v_sum = n = 0
        for s, a, g in sd.minibatches(p, n_passes=1, batch_size=batch_size):
            s = T.tensor(s, device=dev)
            a = T.tensor(a, device=dev).unsqueeze(1)          # (n,) -> (n, 1), like the actor's mean
            g = T.tensor(g, device=dev).unsqueeze(1)

            bc_loss = F.mse_loss(agent.actor(s).mean, a)      # actor: imitate the chosen price
            v_loss  = F.mse_loss(agent.critic(s), g)          # critic: predict the discounted return

            agent.actor.optimizer.zero_grad(); agent.critic.optimizer.zero_grad()
            (bc_loss + 0.5 * v_loss).backward()
            agent.actor.optimizer.step(); agent.critic.optimizer.step()

            bc_sum += bc_loss.item(); v_sum += v_loss.item(); n += 1
        if (e + 1) % log_every == 0:
            print(f'{p:12s} pass {e+1:3d} | actor (BC) loss {bc_sum/n:.4f} | critic loss {v_sum/n:.3f}')

    with T.no_grad():
        agent.actor.actor_log_std.fill_(-1.0)                 # keep some exploration for PPO

for p in products:
    pretrain(p, agents[p])